If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec28_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 28: Designing Experiments — CIs Without the Bootstrap
v.ekc

The payoff of the CLT: confidence intervals straight from a formula — no resampling loop — and the answer to "how many people should I survey?" *before* collecting any data. (Slides: KL Lecture 28 deck.)

**Today's sections:**
1. Review — bootstrap CI for a mean
2. The CLT method
3. The SD of a 0/1 population
4. Try It — choosing a sample size

In [ ]:
import matplotlib
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
import numpy as np
plots.style.use('fivethirtyeight')

---
## 1. Review: Bootstrap CI for a Mean

The Lecture 24 pipeline, for comparison — keep an eye on the interval it produces.

In [ ]:
# original sample
births = Table.read_table('baby.csv')
births.show(3)

In [ ]:
def one_bootstrap_mean():
    resample = births.sample()
    return np.average(resample.column('Maternal Age'))

In [ ]:
# Generate means from 3000 bootstrap samples
num_repetitions = 3000
bstrap_means = make_array()
for i in np.arange(num_repetitions):
    bstrap_means = np.append(bstrap_means, one_bootstrap_mean())

In [ ]:
# Get the endpoints of the 95% confidence interval
left = percentile(2.5, bstrap_means)
right = percentile(97.5, bstrap_means)

In [ ]:
resampled_means = Table().with_columns(
    'Bootstrap Sample Mean', bstrap_means
)
resampled_means.hist(bins=15)
plots.plot([left, right], [0, 0], color='yellow', lw=8);

In [ ]:
left, right

---
## 2. The CLT Method

The CLT says the sample mean is normal with SD = pop SD/√n, and ~95% of a normal curve is within 2 SDs of center. So:

> **95% CI ≈ sample average ± 2 · (pop SD / √n)** — one line of arithmetic. We don't know the pop SD, but the *sample* SD is a good stand-in. (KL deck, slides 13–21.)

In [ ]:
sampled_ages = births.column('Maternal Age')
sample_size = len(sampled_ages)
sample_average = np.average(sampled_ages)
sample_SD = np.std(sampled_ages)
sample_size, sample_average, sample_SD

We need to add and subtract 2 * (population SD)/(sample_size ** 0.5) but we don't have the population SD.

In [ ]:
# Try estimating it from the sample
estimated_SD_of_sample_average = sample_SD / (sample_size**0.5)
estimated_SD_of_sample_average

In [ ]:
# Approximate 95% confidence interval for population mean
sample_average - 2*estimated_SD_of_sample_average, sample_average + 2*estimated_SD_of_sample_average

> **Compare:** nearly identical to the bootstrap interval — for a fraction of the computation. The bootstrap still wins for medians and other statistics the CLT doesn't cover.

---
## 3. The SD of a 0/1 Population

Proportions are just averages of 0/1 data — and 0/1 populations have a beautiful property: their SD is **at most 0.5**, no matter what. (KL deck, slides 24–32.)

In [ ]:
# population of size 10

number_of_ones = 5
zero_one_population = np.append(np.ones(number_of_ones), np.zeros(10 - number_of_ones))

print('Standard Deviation:', np.round(np.std(zero_one_population),2))

zero_one_population

In [ ]:
def sd_of_zero_one_population(number_of_ones):
    """Returns the SD of a population 
    that has 10 elements: num_ones with value 1 and (10 - num_ones) with value 0"""
    zero_one_population = np.append(np.ones(number_of_ones), 
                                    np.zeros(10 - number_of_ones))
    return np.std(zero_one_population)

In [ ]:
possible_ones = np.arange(11)
zero_one_pop = Table().with_columns(
    'Number of Ones', possible_ones,
    'Proportion of Ones', possible_ones / 10
)
zero_one_pop.show()

In [ ]:
sds = zero_one_pop.apply(sd_of_zero_one_population, 'Number of Ones')
zero_one_pop = zero_one_pop.with_column('pop SD', sds)
zero_one_pop.show()

In [ ]:
zero_one_pop.scatter('Proportion of Ones', 'pop SD')

> **The worst case is 0.5** (half ones, half zeros). That means for any proportion, the 95% CI is *at most* `± 2·0.5/√n = ± 1/√n` wide on each side — computable before you collect a single response!

### Try It 1: Design the poll 🗳️

1. You want a 95% confidence interval for a proportion with **total width at most 0.06** (±3 percentage points). What sample size guarantees it, even in the worst case? (*Total width = 4 · 0.5/√n.*)

In [ ]:
# 1. smallest n with 4 * 0.5 / sqrt(n) <= 0.06


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Solve 2/√n ≤ 0.06 → √n ≥ 33.33 → n ≥ 1111.1 → 1112 people. (This is why real polls survey "about 1,100 adults"!)*

```python
# 1.
(2 / 0.06) ** 2          # 1111.1 → need n = 1112
4 * 0.5 / np.sqrt(1112)  # check: ≈ 0.0599 ✓
```

</details>

---
> **Today's takeaway:** CLT turns confidence intervals into arithmetic — mean ± 2·SD/√n — and the 0.5 worst-case SD lets you budget a survey before it exists. This is Homework 9, questions 2 and 3.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Sample size for a proportion
```python
# 95% CI total width <= w   =>   n >= (4 * 0.5 / w) ** 2
```